# Buzzlytics — Model Training (two-stage bee health analysis)

This notebook trains **both** models that power Buzzlytics, end to end:

| Stage | Model | Job | Trained on |
|------|-------|-----|-----------|
| **1. Detector** | YOLOv8n (detection) | Box every bee as `bee` or `pollen_bee` | VnPollenBee |
| **2. Classifier** | YOLOv8n-cls | Decide `healthy` vs `varroa` for each detected bee | VarroaDataset |

### Why two stages (put this in the report)
A single detector cannot reliably find varroa: the *Varroa destructor* mite is a sub-millimetre
speck on the bee's body, and the only public varroa data (VarroaDataset, TU Wien) is **single-bee
classification crops**, not mites boxed inside hive-entrance scenes. Forcing those crops into a
detector as full-frame boxes destroys per-bee localisation and corrupts the `bee` class. So we
follow the standard **detect-then-classify** design used by *IntelliBeeHive* and *BeeAlarmed*:
detect bees first, then classify each detected bee for varroa. At inference a bee classified
`varroa` is relabelled **`varroa_bee`**, giving the three on-screen classes: `bee`, `pollen_bee`,
`varroa_bee`.

### Before you run
Run **`build_dataset_colab.ipynb`** once — it creates `bee_dataset.zip` (detection) and
`varroa_cls.zip` (classification) in your Google Drive. Share-link both and paste below.

> **Runtime → Change runtime type → GPU** (L4 or A100 recommended), then **Run all**.

## 0 · Configuration
Paste the two Drive **share links** from the build step. The hyper-parameters below are the
values to cite in your methodology section.

In [ ]:
# --- Dataset links (Drive 'Anyone with the link') ---
DETECT_URL  = "PASTE_bee_dataset.zip_LINK_HERE"   # stage-1 detection dataset
VARROA_URL  = "PASTE_varroa_cls.zip_LINK_HERE"    # stage-2 classification dataset

# --- Stage-1 detector hyper-parameters ---
DET_EPOCHS = 100      # early-stopped via `patience`, so this is an upper bound
DET_IMGSZ  = 640      # hive-entrance frames are ~1080p; 640 is the YOLO default
DET_BATCH  = 16       # raise to 64 on an L4/A100 (more VRAM headroom)

# --- Stage-2 classifier hyper-parameters ---
CLS_EPOCHS = 30       # crops are simple; converges fast
CLS_IMGSZ  = 128      # crops are small single bees; 128px keeps it cheap
CLS_BATCH  = 64

assert DETECT_URL.startswith("http"), "Paste the bee_dataset.zip share link"
assert VARROA_URL.startswith("http"), "Paste the varroa_cls.zip share link"

## 1 · Confirm GPU
Training on CPU is ~20-40x slower. If this prints `CPU`, switch the runtime to a GPU and re-run.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch Runtime -> GPU!)")

## 2 · Install dependencies
`ultralytics` pins the YOLOv8 version so your results are reproducible (report the exact version);
`gdown` fetches the zips from Drive.

In [ ]:
!pip -q install "ultralytics==8.4.71" gdown pyyaml

---
# Stage 1 — Bee Detector (`bee` / `pollen_bee`)

A YOLOv8n object detector that draws a box around every bee and labels it `bee` or `pollen_bee`.
Trained on **VnPollenBee** (hive-entrance footage, hand-annotated). This is the model that gives
you *individual detection and boxing* of bees.

### 3 · Download + extract the detection dataset
The zip contains `train/ valid/ test/`, each with `images/` + YOLO `labels/`. We auto-locate the
root and print the **class distribution** — quote these counts in your report (note the imbalance:
pollen-bearing bees are rare, which caps `pollen_bee` recall).

In [ ]:
import gdown, zipfile
from pathlib import Path
from collections import Counter

gdown.download(url=DETECT_URL, output="/content/bee_dataset.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("/content/bee_dataset.zip") as z:
    z.extractall("/content/detect")

# Find the dir that directly contains train/images (robust to how the zip nested folders).
DATA_DIR = next((str(p.parent.parent) for p in Path("/content/detect").rglob("train/images")), None)
assert DATA_DIR, "No train/images in the zip. Re-build with build_dataset_colab.ipynb."
print("DATA_DIR:", DATA_DIR)

# Class distribution (0=bee, 1=pollen_bee) + image counts per split.
counts = Counter()
for txt in Path(DATA_DIR).rglob("labels/*.txt"):
    for line in txt.read_text().splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1
print("instances per class (0=bee, 1=pollen_bee):", dict(sorted(counts.items())))
for split in ["train", "valid", "test"]:
    d = Path(DATA_DIR) / split / "images"
    print(f"  {split}: {len(list(d.glob('*'))) if d.is_dir() else 0} images")

### 4 · Write the dataset config (`data.yaml`)
YOLO reads class names + split paths from this file. Two classes only — varroa is **not** here
because it is handled by the stage-2 classifier.

In [ ]:
import yaml
data_cfg = {
    "path": DATA_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 2,
    "names": {0: "bee", 1: "pollen_bee"},
}
with open("/content/bee_dataset.yaml", "w") as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False)
print(open("/content/bee_dataset.yaml").read())

### 5 · Fine-tune the detector
Starts from COCO-pretrained `yolov8n.pt` (transfer learning) and fine-tunes on bees. The
augmentations (HSV jitter, horizontal flip, mosaic, small rotation) improve robustness to lighting
and bee orientation — **list them in your methodology**. `patience=25` early-stops when validation
mAP plateaus, and `seed=42` makes the run reproducible.

In [ ]:
from ultralytics import YOLO
det_model = YOLO("yolov8n.pt")
det_model.train(
    data="/content/bee_dataset.yaml",
    epochs=DET_EPOCHS, imgsz=DET_IMGSZ, batch=DET_BATCH,
    name="bee_detector", patience=25, seed=42,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # colour jitter
    fliplr=0.5, mosaic=1.0, degrees=5.0, # geometric augmentation
)

### 6 · Evaluate the detector (results table)
Reports mAP@50 and mAP@50-95 on the held-out **test** split, plus **per-class** mAP. Put this table
in your results section. Expect `pollen_bee` < `bee` because pollen-bearing examples are sparse.

In [ ]:
det_best = "runs/detect/bee_detector/weights/best.pt"
m = YOLO(det_best).val(data="/content/bee_dataset.yaml", split="test")
print("mAP@50:    ", round(float(m.box.map50), 4))
print("mAP@50-95: ", round(float(m.box.map), 4))
det_names = {0: "bee", 1: "pollen_bee"}
print("\nPer-class mAP@50:")
for i, ap in zip(m.box.ap_class_index, m.box.ap50):
    print(f"  {det_names.get(int(i), i):<12} {float(ap):.4f}")

---
# Stage 2 — Varroa Classifier (`healthy` / `varroa`)

A YOLOv8n **classification** model. At inference the pipeline crops each bee the detector found and
runs this model; a `varroa` verdict flips that box to `varroa_bee`. Trained on **VarroaDataset**
close-up single-bee crops, where the mite is actually visible.

### 7 · Download + extract the classification dataset
YOLOv8-cls expects an **ImageFolder**: `train/ val/ test/`, each with `healthy/` and `varroa/`
sub-folders. We print per-class counts — note the imbalance (~2.4 healthy : 1 varroa), which we
capped during the build to keep it balanced enough to learn varroa.

In [ ]:
gdown.download(url=VARROA_URL, output="/content/varroa_cls.zip", quiet=False, fuzzy=True)
with zipfile.ZipFile("/content/varroa_cls.zip") as z:
    z.extractall("/content/varroa")

CLS_DIR = next((str(p.parent.parent) for p in Path("/content/varroa").rglob("train/varroa")), None)
assert CLS_DIR, "No train/varroa folder. Re-build with build_dataset_colab.ipynb."
print("CLS_DIR:", CLS_DIR)
for split in ["train", "val", "test"]:
    for lab in ["healthy", "varroa"]:
        d = Path(CLS_DIR) / split / lab
        print(f"  {split}/{lab}: {len(list(d.glob('*'))) if d.is_dir() else 0}")

### 8 · Train the varroa classifier
Fine-tunes `yolov8n-cls.pt` (ImageNet-pretrained). Small images + a simple binary task → trains in
a few minutes. `patience=10` early-stops on plateau.

In [ ]:
cls_model = YOLO("yolov8n-cls.pt")
cls_model.train(
    data=CLS_DIR,
    epochs=CLS_EPOCHS, imgsz=CLS_IMGSZ, batch=CLS_BATCH,
    name="varroa_cls", patience=10, seed=42,
)

### 9 · Evaluate the classifier
Top-1 accuracy on the test split. For the report, also mention that varroa is the **minority class**,
so per-class recall (catching infected bees) matters more than raw accuracy — a colony-health system
should err toward flagging.

In [ ]:
cls_best = "runs/classify/varroa_cls/weights/best.pt"
cm = YOLO(cls_best).val(data=CLS_DIR, split="test")
print("top-1 accuracy:", round(float(cm.top1), 4))
print("top-5 accuracy:", round(float(cm.top5), 4))

---
## 10 · Download both trained models
Place them in your repo so the pipeline picks them up automatically:

| File | Put it here |
|------|-------------|
| `best.pt` (detector) | `cv_pipeline/weights/best.pt` |
| `varroa_cls.pt` (classifier) | `cv_pipeline/weights/varroa_cls.pt` |

The backend reads `config.yaml` → loads both. If `varroa_cls.pt` is absent the app still runs
(detection only, varroa inert) — no crash.

In [ ]:
import shutil
from google.colab import files
shutil.copy(cls_best, "/content/varroa_cls.pt")
files.download(det_best)              # -> cv_pipeline/weights/best.pt
files.download("/content/varroa_cls.pt")  # -> cv_pipeline/weights/varroa_cls.pt